# DS02 · Participant identity, tidy tables, and join cardinality

<!-- paper-first -->
### Begin with the paper question

**Read or revisit [PD02](../../curriculum/papers/design.md#pd02).** Use the assigned first-pass sections in the guide; if you already read them, return only to the relevant figure or claim. Do this before the technical explanation below.

**Motivation:** How could an incorrect table join change the number of apparent people and the estimated association?

Write a two-sentence prediction and one thing you cannot yet explain. Ask your AI tutor to locate evidence in the supplied paper and distinguish it from inference. A paper link motivates this question; it does not mean the paper uses every method demonstrated here.

**After the experiment:** revisit your prediction in the [evidence ledger](../../curriculum/coursework/EVIDENCE_LEDGER.md). Explain one mechanism you now understand, cite a result from this notebook, and name a paper claim this exercise still cannot test. Keep a small demonstration distinct from a reproduction of the study.
<!-- /paper-first -->

**Format:** 75–100 minutes of guided work, plus 30–60 minutes in the assigned existing course material. **Prerequisite:** the foundations notebooks; follow this strand in order. The core exercise is synthetic, offline, and independently runnable. It demonstrates mechanics, not a validated participant-data analysis.

## Existing course material

Read [Data 8: Joining tables by columns](https://github.com/data-8/textbook/blob/5235b7653f8dfaeb90e43419b9aa069322f2d60b/chapters/08/4/Joining_Tables_by_Columns.ipynb). Use the indicated topic, then return here to apply it to a neuroimaging question. Berkeley material is linked in its original form, not adapted or redistributed; its CC BY-NC-ND terms remain upstream. Neuromatch material is CC BY 4.0 with separately licensed software; selected unmodified copies live in `third_party/data_science`. The explanation and dataset below are original.

## Understand the transformation

The key to a longitudinal imaging table is often participant plus visit, rather than participant alone. A separate scanner or behavioral table may use different row order. Joining on row position can make a plausible table with the wrong scientific pairings. Joining repeated records on participant alone can instead create a many-to-many expansion, giving artificial weight to people with more visits.

A tidy table makes the observation explicit: one row might mean one participant-visit-ROI combination. A data dictionary states units, allowable values, missingness, and the key. Converting from wide to long format changes how observations are stored; it does not create new independent participants. An AI-generated groupby must name the intended weighting rule, because averaging all rows can weight frequently scanned participants more heavily.

We build two visits per participant and deliberately shuffle the outcome table. A protected merge on both ID and visit should recover the correct pairings without changing row count. The incorrect join will produce twice as many rows. Instead of suppressing a merge warning, trace the duplicated keys. On real projects also retain unmatched IDs and reasons for exclusion. A successful join must preserve the intended sample, not merely produce no error.

## AI-guided prediction

First answer in your own words; then send this to Goose/Ollama or ChatGPT:

> Ask me what one row represents and whether participant_id is unique. Propose an explicit composite-key merge, validate its cardinality, and display unmatched records. Do not join by row position or silently drop duplicated records.

Use the model as a tutor and snippet writer. Require it to name the axes, units, fitting population, expected output, and one failure check. A code cell that runs is not proof that it answers the scientific question. Keep raw data unchanged and save your actual settings.

## Experiment

Predict the correct and incorrect row counts. Compare ID-only joining with ID-and-visit joining. Deliberate error: interpret the expanded table as more participants.

Run the following cells in order. Before each, predict what should remain unchanged and what should differ. The assertions test specific mathematical or bookkeeping properties, not clinical validity.

In [1]:
import pandas as pd
scans = pd.DataFrame({'id':['a','a','b','b'], 'visit':[1,2,1,2], 'roi':[10,11,20,22]})
behavior = pd.DataFrame({'id':['b','a','b','a'], 'visit':[2,1,1,2], 'score':[202,101,201,102]})
good = scans.merge(behavior, on=['id','visit'], validate='one_to_one', indicator=True)
bad = scans.merge(behavior, on='id')
assert len(good) == 4 and len(bad) == 8
assert good['score'].tolist() == [101,102,201,202]
assert good['id'].nunique() == bad['id'].nunique() == 2
print(good.to_string(index=False))
print('Bad row count:', len(bad))

id  visit  roi  score _merge
 a      1   10    101   both
 a      2   11    102   both
 b      1   20    201   both
 b      2   22    202   both
Bad row count: 8


## Explain, break, transfer

1. Save an input → operation → output diagram and state what information was lost.
2. Make the specified wrong choice above. Compare its result with the reference checks; explain why the misleading result is possible.
3. Work through the assigned upstream chapter's example using its own environment or hosted reader. Record one difference between its data and a participant/voxel/time-series dataset.
4. Ask the AI for a short application to a real imaging table, but do not run it until participant identifiers, units, missingness, and any training/test boundary are explicit. Never infer those properties from the column names alone.

**Evidence to submit:** one labeled result, the changed parameter, a failure diagnosis, and a five-sentence interpretation that separates a computational check from the research claim. Explain the result without looking at the model's wording.

<details><summary>Instructor check / answer guide</summary>

The correct table has four rows; the ID-only join has eight. Unique people remain two in both. Duplicate rows are not extra independent information.

</details>

**Scope:** This local notebook and its numerical checks are part of the executable core. Completion of the external chapter is a learner assignment; its execution is not implied by the local result. No endorsement by the source authors or USC is implied.

### Return to the research question

Reopen [PD02](../../curriculum/papers/design.md#pd02) and your initial two-sentence prediction. In your [evidence ledger](../../curriculum/coursework/EVIDENCE_LEDGER.md):

1. Cite one output or diagnostic from this lesson and explain the transformation it demonstrates.
2. Revise one claim or question from the paper, with a figure/section locator. State what this small exercise still cannot establish about the published result.
3. Ask AI to propose a next check. Accept, revise or reject it with a scientific reason. Then explain your decision aloud without reading the AI response.

Reuse this entry in the A2 portfolio when relevant; a separate report is unnecessary.
